# Coverage analysis

- Bins ribo reads to CDSs
- Generates metagene

In [1]:
import json
from pathlib import Path

import pandas as pd

from src.helpers.coverage_helper import bin_ribo_counts, build_metagene, plot_metagene

Set the metagenome id and ribo id to your chosen sample.

In [3]:
META_ID = "META_ID"
RIBO_ID = "RIBO_ID"
MANIFEST_PATH = Path("file_manifest.json")
manifest = json.loads(MANIFEST_PATH.read_text())
ribo_bam = manifest["ribo"]["alignment"][RIBO_ID]
master_path = manifest["metagenome"]["master_sheets"][META_ID]
coverage_dir = Path("output/coverage") / RIBO_ID
master_df = pd.read_csv(master_path, sep="\t")

master_df.set_index("CDS_id", inplace=True)

Below are the metagene-specific parameters.

In [4]:
BINNING_METHOD = "overlap"  # or 5prime, 3prime, center
ANCHOR = "5prime"  # or 3prime, center
WINDOW = 100
READ_LENGTH = 28

Bin Ribo-ITP counts

In [6]:
master_df['binned_counts'] = bin_ribo_counts(ribo_bam, master_df, BINNING_METHOD)

Build metagene

In [7]:
metagene_df = build_metagene(ribo_bam, master_df, ANCHOR, WINDOW, READ_LENGTH)

Plot metagene

In [8]:
meta_fig = plot_metagene(metagene_df, ANCHOR, WINDOW, f"{RIBO_ID} {ANCHOR} metagene")
meta_fig.show()

Save results

In [9]:
coverage_dir.mkdir(parents=True, exist_ok=True)
master_df.to_csv(coverage_dir / f"{RIBO_ID}_cds_counts.tsv", sep="\t", index=True)
metagene_df.to_csv(coverage_dir / f"metagene_{ANCHOR}.tsv", sep="\t", index=False)